# 01 · Training — Dataset Assembly & Train / Val / Test Splits

Merge real faces and curated face-swaps into a single DataFrame. Build train (≈1:1 balance), val, and test (99:1 real:fake imbalance) splits grouped by `target_id` to prevent context leakage.

In [2]:
import pandas as pd
import glob
import json
import os
from tqdm import tqdm

#### Read Bad Images from Manual Review

In [2]:
with open('./08 Manual Review/celeba_bad_images.json', 'rb') as file:
    celeba_bads = json.load(file)

with open('./08 Manual Review/flickr_bad_images.json', 'rb') as file:
    flickr_bads = json.load(file)

with open('./08 Manual Review/utk_bad_images.json', 'rb') as file:
    utk_bads = json.load(file)

with open('./08 Manual Review/vgg_bad_images.json', 'rb') as file:
    vgg_bads = json.load(file)

In [3]:
def jpg_to_png(df):
    df['source'] = df['source'].apply(lambda x: x.replace('.jpg','.png'))
    df['target_p50'] = df['target_p50'].apply(lambda x: x.replace('.jpg','.png'))
    df['target_p99'] = df['target_p99'].apply(lambda x: x.replace('.jpg','.png'))
    return df

In [4]:
flickr = pd.read_parquet('./04 Datasets/flickr_swap_dataset.parquet')
flickr = jpg_to_png(flickr)
flickr_cleaned = flickr[~(flickr['source'].isin(flickr_bads)) & ~(flickr['target_p50'].isin(flickr_bads)) & ~(flickr['target_p99'].isin(flickr_bads))]

In [5]:
celeba = pd.read_parquet('./04 Datasets/celeba_swap_dataset.parquet')
celeba = jpg_to_png(celeba)
celeba_cleaned = celeba[~(celeba['source'].isin(celeba_bads)) & ~(celeba['target_p50'].isin(celeba_bads)) & ~(celeba['target_p99'].isin(celeba_bads))]

In [6]:
utk = pd.read_parquet('./04 Datasets/utk_swap_dataset.parquet')
utk = jpg_to_png(utk)
utk_cleaned = utk[~(utk['source'].isin(utk_bads)) & ~(utk['target_p50'].isin(utk_bads)) & ~(utk['target_p99'].isin(utk_bads))]

In [7]:
vgg = pd.read_parquet('./04 Datasets/vgg_swap_dataset.parquet')
vgg = jpg_to_png(vgg)
vgg_cleaned = vgg[~(vgg['source'].isin(vgg_bads)) & ~(vgg['target_p50'].isin(vgg_bads)) & ~(vgg['target_p99'].isin(vgg_bads))]

In [8]:
print(f"""
Flickr: {len(flickr)}
Flickr Cleaned: {len(flickr_cleaned)}
Removed: {len(flickr) - len(flickr_cleaned)} |  {round((len(flickr) - len(flickr_cleaned)) / len(flickr),2)}
-------------------------
Celeba: {len(celeba)}
Celeba Cleaned: {len(celeba_cleaned)}
Removed: {len(celeba) - len(celeba_cleaned)} |  {round((len(celeba) - len(celeba_cleaned)) / len(celeba),2)}
-------------------------
UTK: {len(utk)}
UTK Cleaned: {len(utk_cleaned)}
Removed: {len(utk) - len(utk_cleaned)} |  {round((len(utk) - len(utk_cleaned)) / len(utk),2)}
-------------------------
VGG: {len(vgg)}
VGG Cleaned: {len(vgg_cleaned)}
Removed: {len(vgg) - len(vgg_cleaned)} |  {round((len(vgg) - len(vgg_cleaned)) / len(vgg),2)}
-------------------------
""")


Flickr: 13751
Flickr Cleaned: 10415
Removed: 3336 |  0.24
-------------------------
Celeba: 16813
Celeba Cleaned: 14435
Removed: 2378 |  0.14
-------------------------
UTK: 2783
UTK Cleaned: 2374
Removed: 409 |  0.15
-------------------------
VGG: 416
VGG Cleaned: 159
Removed: 257 |  0.62
-------------------------



In [9]:
dataset = pd.concat([flickr_cleaned, celeba_cleaned, utk_cleaned, vgg_cleaned])
len(dataset)

27383

In [10]:
rows = []
for record in dataset.to_dict(orient = 'records'):
    record_temp = {
        'dataset_name' : record['dataset_name'],
        'path' : record['source'],
        'label' : 0
    }
    rows.append(record_temp)
    record_temp = {
        'dataset_name' : record['dataset_name'],
        'path' : record['target_p50'],
        'label' : 0
    }
    rows.append(record_temp)
    record_temp = {
        'dataset_name' : record['dataset_name'],
        'path' : record['target_p99'],
        'label' : 0
    }
    rows.append(record_temp)
    record_temp = {
        'dataset_name' : record['dataset_name'],
        'path' : record['source'].replace('.png','') + '_' + record['target_p50'],
        'label' : 1
    }
    rows.append(record_temp)
    record_temp = {
        'dataset_name' : record['dataset_name'],
        'path' : record['source'].replace('.png','') + '_' + record['target_p99'],
        'label' : 1
    }
    rows.append(record_temp)
final_dataset = pd.DataFrame(rows)

In [11]:
final_dataset.head(1)

,dataset_name,path,label
0,FLICKR,10777.png,0


In [12]:
print(len(final_dataset))
final_dataset.drop_duplicates(subset = ['path'], keep = 'first', inplace = True)
print(len(final_dataset))

136915
85057


In [13]:
final_dataset['label'].value_counts(normalize = False)

label
1    54766
0    30291
Name: count, dtype: int64

In [82]:
final_dataset.to_parquet('./04 Datasets/final_dataset.parquet')

In [81]:
final_dataset.tail()

,dataset_name,path,label
30286,CELEBA,092438_raw.png,0
30287,CELEBA,135461_raw.png,0
30288,CELEBA,017385_raw.png,0
30289,CELEBA,181586_raw.png,0
30290,CELEBA,051127_raw.png,0


In [19]:
len(final_dataset)

85057

In [75]:
all_data = glob.glob('./03 All Data/*.png', recursive = True)
all_data_name = [os.path.basename(i) for i in glob.glob('./03 All Data/*.png', recursive = True)]

In [78]:
final_dataset = final_dataset[final_dataset['path'].isin(all_data_name)]

In [79]:
len(final_dataset)

115347

In [ ]:
from melitk.fda_artifacts.artifacts import BytesArtifact
import os
# os.environ['TIGER_TOKEN'] = ''
artifact = BytesArtifact(
    name="deepfakes_annotatons",
    version="0.0.1",
    ttl=30*12,
    confidentiality="internal",
    integrity="low",
    availability="low",
)
artifact = artifact.upload(path = './04 Datasets/final_dataset.parquet')

In [31]:
final_dataset['label'].value_counts()

label
1    54766
0    30290
Name: count, dtype: int64

In [55]:
final_dataset['dataset_name'].value_counts()

dataset_name
CELEBA      44453
FLICKR      32721
UTK_FACE     7328
VGG_FACE      554
Name: count, dtype: int64

In [56]:
raws_info = pd.read_parquet('raws_info.parquet')

In [57]:
final_dataset.head()

,dataset_name,path,label
0,FLICKR,10777.png,0
1,FLICKR,14659.png,0
2,FLICKR,31199.png,0
3,FLICKR,10777_14659.png,1
4,FLICKR,10777_31199.png,1


In [58]:
raws_info.head()

,path,dataset_name,label
0,0242_04_raw.png,VGG_FACE,0
1,0110_09_raw.png,VGG_FACE,0
2,0565_01_raw.png,VGG_FACE,0
3,0101_07_raw.png,VGG_FACE,0
4,0041_03_raw.png,VGG_FACE,0


In [61]:
final_dataset = pd.concat([final_dataset, raws_info])
final_dataset[final_dataset['path'].duplicated()]

,dataset_name,path,label


In [62]:
final_dataset.label.value_counts()

label
0    60581
1    54766
Name: count, dtype: int64

In [63]:
final_dataset.head()

,dataset_name,path,label
0,FLICKR,10777.png,0
1,FLICKR,14659.png,0
2,FLICKR,31199.png,0
3,FLICKR,10777_14659.png,1
4,FLICKR,10777_31199.png,1


In [67]:
deepfakes_dataset = inventory.get(visible_id='14c9acc1-3d2d-4e28-a11e-13fbde39fe9a')
deepfakes_dataset.download('./deepfakes_dataset.7z')

2026-05-28T03:31:09+0000 - INFO: Identifying if token is present and is valid.  [level: INFO]
2026-05-28T03:31:09+0000 - INFO: Headers: X-Tiger-Token  [level: INFO]
2026-05-28T03:31:09+0000 - DEBUG: Getting artifact.  [level: DEBUG]
2026-05-28T03:31:09+0000 - DEBUG: Downloading using single-file mechanism.  [level: DEBUG]
2026-05-28T03:31:09+0000 - INFO: Identifying if token is present and is valid.  [level: INFO]
Metric name must only contains ASCII alphanumerics, underscores, and periods.
2026-05-28T03:31:09+0000 - DEBUG: Getting url to download an artifact.  [level: DEBUG]
2026-05-28T03:31:10+0000 - DEBUG: Open signed URL: https://7e6fe3c1-artifactsservi-legacyprodlow-useast1-default-9750500.s3.amazonaws.com/14c9acc1-3d2d-4e28-a11e-13fbde39fe9a/0.0.1.bin  [level: DEBUG]
2026-05-28T03:31:10+0000 - INFO: Compression? no  [level: INFO]
Metric name must only contains ASCII alphanumerics, underscores, and periods.


In [8]:
def split_dataset(df_path, target_pos_per_split=100, seed=42):
    """
    Divide el dataset en train (50/50) / val / test (1% positivas cada uno).
    Agrupa por target_id para evitar leakage de contexto.
    """
    df = pd.read_parquet(df_path)

    def get_target(row):
        name = row['path'].rsplit('.', 1)[0]
        return name.split('_')[0] if row['label'] == 1 else name

    df['target_id'] = df.apply(get_target, axis=1)
    df['group'] = df['dataset_name'] + '::' + df['target_id']

    stats = df.groupby('group').agg(
        n_pos=('label', lambda x: (x == 1).sum()),
        n_neg=('label', lambda x: (x == 0).sum()),
    ).reset_index()

    pos_groups = stats[stats['n_pos'] > 0].sample(frac=1, random_state=seed).reset_index(drop=True)
    neg_groups = stats[stats['n_pos'] == 0].sample(frac=1, random_state=seed).reset_index(drop=True)

    # Repartir grupos-con-positiva alternando entre val y test
    val_g, test_g = [], []
    v_pos = t_pos = 0
    for _, row in pos_groups.iterrows():
        if v_pos >= target_pos_per_split and t_pos >= target_pos_per_split:
            break
        if v_pos <= t_pos and v_pos < target_pos_per_split:
            val_g.append(row['group']); v_pos += row['n_pos']
        else:
            test_g.append(row['group']); t_pos += row['n_pos']

    val_neg_needed = max(0, 99 * v_pos - stats[stats['group'].isin(val_g)]['n_neg'].sum())
    test_neg_needed = max(0, 99 * t_pos - stats[stats['group'].isin(test_g)]['n_neg'].sum())

    # Repartir grupos solo-negativa proporcionalmente entre val y test
    val_neg_g, test_neg_g = [], []
    v_neg = t_neg = 0
    for _, row in neg_groups.iterrows():
        if v_neg >= val_neg_needed and t_neg >= test_neg_needed:
            break
        v_ratio = v_neg / val_neg_needed if val_neg_needed > 0 else 1
        t_ratio = t_neg / test_neg_needed if test_neg_needed > 0 else 1
        if v_ratio <= t_ratio and v_neg < val_neg_needed:
            val_neg_g.append(row['group']); v_neg += row['n_neg']
        elif t_neg < test_neg_needed:
            test_neg_g.append(row['group']); t_neg += row['n_neg']
        else:
            val_neg_g.append(row['group']); v_neg += row['n_neg']

    val_keys = set(val_g) | set(val_neg_g)
    test_keys = set(test_g) | set(test_neg_g)

    def assign(g):
        if g in val_keys: return 'val'
        if g in test_keys: return 'test'
        return 'train'

    df['split'] = df['group'].apply(assign)

    def trim_to_1pct(split_df, sub_seed):
        pos = split_df[split_df['label'] == 1]
        neg_all = split_df[split_df['label'] == 0]
        n_neg = min(99 * len(pos), len(neg_all))
        neg = neg_all.sample(n=n_neg, random_state=sub_seed)
        return pd.concat([pos, neg], ignore_index=True)

    val = trim_to_1pct(df[df['split'] == 'val'], seed)
    test = trim_to_1pct(df[df['split'] == 'test'], seed + 1)

    tr = df[df['split'] == 'train']
    n = min((tr['label'] == 1).sum(), (tr['label'] == 0).sum())
    train = pd.concat([
        tr[tr['label'] == 1].sample(n=n, random_state=seed),
        tr[tr['label'] == 0].sample(n=n, random_state=seed + 2),
    ], ignore_index=True)

    cols = ['dataset_name', 'path', 'label']
    return train[cols], val[cols], test[cols]

#### Splits

In [9]:
train, val, test = split_dataset('./02 Datasets/final_dataset.parquet', target_pos_per_split=100, seed=42)

for name, sdf in [('Train', train), ('Val', val), ('Test', test)]:
    pos = (sdf['label'] == 1).sum()
    print(f"{name}: {len(sdf):,} ({pos:,} pos / {len(sdf)-pos:,} neg, {100*pos/len(sdf):.2f}% pos)")

train.to_parquet('./07 Splits/train.parquet', index=False)
val.to_parquet('./07 Splits/val.parquet', index=False)
test.to_parquet('./07 Splits/test.parquet', index=False)

Train: 78,394 (39,197 pos / 39,197 neg, 50.00% pos)
Val: 11,600 (116 pos / 11,484 neg, 1.00% pos)
Test: 10,000 (100 pos / 9,900 neg, 1.00% pos)


In [10]:
train.to_parquet('./07 Splits/train.parquet', index=False)
val.to_parquet('./07 Splits/val.parquet', index=False)
test.to_parquet('./07 Splits/test.parquet', index=False)

In [ ]:
# Train: 49,578 (24,789 pos / 24,789 neg, 50.00% pos)
# Val: 3,054 (116 pos / 2,938 neg, 3.80% pos)
# Test: 2,663 (100 pos / 2,563 neg, 3.76% pos)